# Problem 3: 训练工具 (Training Utilities)

在本问题中，你将实现训练神经网络所需的工具函数，
包括损失函数、梯度裁剪、优化器和学习率调度。

## 3.1 交叉熵损失 (Cross-Entropy Loss)

**目标**: 实现 `run_cross_entropy` 函数。

交叉熵损失用于分类任务，衡量预测分布与真实标签之间的差异。

**数学定义**:

对于单个样本，交叉熵损失为：

$$\text{CE}(y, \hat{y}) = -\log\left(\frac{\exp(\hat{y}_y)}{\sum_{j=1}^{K} \exp(\hat{y}_j)}\right) = -\hat{y}_y + \log\left(\sum_{j=1}^{K} \exp(\hat{y}_j)\right)$$

其中：
- $y \in \{0, 1, ..., K-1\}$ 是真实类别索引
- $\hat{y} \in \mathbb{R}^K$ 是未归一化的 logits（预测值）
- $K$ 是类别数量（vocab_size）

对于批量样本，我们计算平均损失：

$$\text{Loss} = \frac{1}{N}\sum_{i=1}^{N} \text{CE}(y_i, \hat{y}_i)$$

**实现要求**:
- 不能使用 `torch.nn.CrossEntropyLoss` 或 `F.cross_entropy`
- 需要数值稳定性（使用 log-sum-exp 技巧）
- 返回平均损失（标量张量）

**数值稳定性提示**:

使用 log-sum-exp 技巧：
$$\log\left(\sum_{j} \exp(\hat{y}_j)\right) = \max(\hat{y}) + \log\left(\sum_{j} \exp(\hat{y}_j - \max(\hat{y}))\right)$$

**对应函数**: `tests/adapters.py` 中的 `run_cross_entropy(inputs, targets)`

**参数**:
- `inputs`: $[batch\_size, vocab\_size]$ 的 logits
- `targets`: $[batch\_size]$ 的目标类别索引
- 返回: 标量损失值

## 3.2 梯度裁剪 (Gradient Clipping)

**目标**: 实现 `run_gradient_clipping` 函数。

梯度裁剪用于防止训练过程中的梯度爆炸问题。

**裁剪策略**:

给定一组参数，计算所有参数梯度的全局 L2 范数：

$$g_{global} = \sqrt{\sum_{p \in \text{params}} \|\nabla p\|_2^2}$$

如果 $g_{global} > \text{max\_l2\_norm}$，则将所有参数的梯度按比例缩放：

$$\nabla p \leftarrow \nabla p \cdot \frac{\text{max\_l2\_norm}}{g_{global}}$$

**实现要求**:
- **就地修改**参数的梯度（`parameter.grad`）
- 计算所有参数梯度的全局 L2 范数
- 如果超过阈值，按比例缩放所有梯度
- 返回 None（函数直接修改参数）

**对应函数**: `tests/adapters.py` 中的 `run_gradient_clipping(parameters, max_l2_norm)`

**提示**: 使用 `torch.nn.utils.clip_grad_norm_` 是不允许的，
你需要自己计算梯度范数并进行缩放。

## 3.3 AdamW 优化器 (AdamW Optimizer)

**目标**: 实现 `get_adamw_cls` 函数，返回一个 AdamW 优化器类。

**参考**: [Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101) - "Decoupled Weight Decay Regularization"

**Adam vs AdamW**:

AdamW 是 Adam 的改进版本，主要区别在于**权重衰减**的实现方式：

- **Adam**: 权重衰减与梯度更新耦合在一起
- **AdamW**: 权重衰减直接应用于参数，与梯度更新解耦

**AdamW 更新规则**:

对于每个参数 $\theta$：

1. **计算梯度**: $g_t = \nabla_{\theta} f(\theta_{t-1})$

2. **更新一阶矩估计**: $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$

3. **更新二阶矩估计**: $v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$

4. **偏差修正**: $\hat{m}_t = \frac{m_t}{1-\beta_1^t}$, $\hat{v}_t = \frac{v_t}{1-\beta_2^t}$

5. **权重衰减**: $\theta_t = \theta_{t-1} - \eta \lambda \theta_{t-1}$

6. **参数更新**: $\theta_t = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

其中：
- $\eta$ 是学习率
- $\beta_1, \beta_2$ 是矩估计的指数衰减率（通常 $\beta_1=0.9, \beta_2=0.999$）
- $\lambda$ 是权重衰减系数
- $\epsilon$ 是数值稳定常数（通常 $10^{-8}$）

**实现要求**:
- 返回一个 PyTorch 优化器类（不是实例）
- 可以返回 `torch.optim.AdamW`（这是允许的）
- 或者你自己实现一个继承自 `torch.optim.Optimizer` 的类

**对应函数**: `tests/adapters.py` 中的 `get_adamw_cls()`

**注意**: 这个函数返回的是**类**，不是实例。测试代码会这样使用：
```python
adamw_cls = get_adamw_cls()
optimizer = adamw_cls(model.parameters(), lr=1e-3, weight_decay=0.01)
```

## 3.4 带预热的余弦学习率调度 (Cosine Learning Rate Schedule with Warmup)

**目标**: 实现 `run_get_lr_cosine_schedule` 函数。

学习率调度在训练过程中动态调整学习率，有助于模型收敛。

**调度器定义**:

给定参数：
- $\alpha_{max}$: 最大学习率
- $\alpha_{min}$: 最小学习率
- $T_w$: 预热迭代数
- $T_c$: 余弦周期迭代数
- $t$: 当前迭代数

**阶段 1: 预热阶段** ($0 \leq t < T_w$)

学习率线性地从 0 增加到 $\alpha_{max}$：

$$\alpha(t) = \frac{\alpha_{max}}{T_w} \cdot t$$

**阶段 2: 余弦退火阶段** ($T_w \leq t < T_w + T_c$)

学习率按余弦曲线从 $\alpha_{max}$ 衰减到 $\alpha_{min}$：

$$\alpha(t) = \alpha_{min} + \frac{1}{2}(\alpha_{max} - \alpha_{min})\left(1 + \cos\left(\pi \cdot \frac{t - T_w}{T_c}\right)\right)$$

**阶段 3: 超过周期后** ($t \geq T_w + T_c$)

学习率保持在最小值：

$$\alpha(t) = \alpha_{min}$$

**可视化**:

学习率随迭代数的变化曲线：
- 预热阶段：线性上升
- 余弦阶段：平滑下降（余弦曲线）
- 之后：保持最小值

**实现要求**:
- 根据当前迭代数 $t$ 计算对应的学习率
- 正确处理三个阶段
- 返回一个浮点数（学习率值）

**对应函数**: `tests/adapters.py` 中的 `run_get_lr_cosine_schedule(it, max_learning_rate, min_learning_rate, warmup_iters, cosine_cycle_iters)`

**示例**:
```python
# 参数: max_lr=1e-3, min_lr=1e-5, warmup=100, cycle=1000
lr_0 = run_get_lr_cosine_schedule(0, 1e-3, 1e-5, 100, 1000)    # 0.0 (预热开始)
lr_50 = run_get_lr_cosine_schedule(50, 1e-3, 1e-5, 100, 1000)   # 5e-4 (预热中)
lr_100 = run_get_lr_cosine_schedule(100, 1e-3, 1e-5, 100, 1000)  # 1e-3 (预热结束，余弦开始)
lr_600 = run_get_lr_cosine_schedule(600, 1e-3, 1e-5, 100, 1000)  # ~5e-4 (余弦中)
lr_1100 = run_get_lr_cosine_schedule(1100, 1e-3, 1e-5, 100, 1000) # 1e-5 (周期结束)
```